In [1]:
import torch
import os
import sys
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
# Project path
sys.path.append(os.path.abspath("../data"))
from models.gcn_model import GCN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [3]:
def test_global_model(global_model, graph):
    """
    Test global model on a hospital's local graph
    """
    global_model.eval()
    with torch.no_grad():
        out = global_model(graph.x.to(device), graph.edge_index.to(device))
        preds = out.argmax(dim=1).cpu().numpy()
        labels = graph.y.cpu().numpy()

    acc = accuracy_score(labels, preds)
    return acc

In [4]:
if __name__ == "__main__":

    # Load Hospital A graph (test data)
    graph_A = torch.load("../data/graph/graph_A.pt", weights_only=False)
    graph_A = graph_A.to(device)

    input_dim = graph_A.num_node_features

    # Load global model
    global_model = GCN(input_dim, 32, 2).to(device)
    global_model.load_state_dict(
        torch.load("../data/models/global_model.pth", map_location=device)
    )

    print("Global model loaded")

    # Test on Hospital A
    accuracy = test_global_model(global_model, graph_A)

    print(f"Global Model Accuracy on Hospital A: {accuracy:.4f}")

Global model loaded
Global Model Accuracy on Hospital A: 0.9844
